# 05: Clean NWSL Media Coverage Data

Loads the raw MediaCloud pull and filters it down to articles actually relevant to the NWSL: regex-matches on title (league name, team names/nicknames, "women's soccer"). Also scores each article's sentiment with VADER (positive/negative/neutral split).

Also cleans the YouTube channel/video pull from `04`: parses dates, derives publish year/month for the video-level data, and leaves the channel-level snapshot mostly as-is (it's already one tidy row per channel).

**Inputs:** `data/raw/nwsl_articles_raw.csv`, `data/raw/youtube_channels_raw.csv`, `data/raw/youtube_videos_raw.csv`
**Outputs:** `data/processed/nwsl_articles_filtered.csv`, `data/processed/nwsl_articles_with_sentiment.csv`, `data/processed/youtube_channels.csv`, `data/processed/youtube_videos.csv`

In [1]:
import pandas as pd
import re
import os

DATA_RAW_DIR = os.path.join("..", "data", "raw")
DATA_PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)

KEYWORDS = [
    "NWSL",
    "National Women's Soccer League",
    "women's soccer",
    "Angel City",
    "OL Reign",
    "Portland Thorns",
    "North Carolina Courage",
    "Washington Spirit",
    "Chicago Red Stars",
    "Houston Dash",
    "Gotham FC",
    "Orlando Pride",
    "Racing Louisville FC",
    "San Diego Wave",
    "Boston Legacy FC",
    "Denver Summit FC",
    "Sky Blue FC",
    "Boston Breakers",
]

# \b...\b word-boundary anchors around the whole alternation, so e.g. "NWSL" only
# matches as its own word and not as a substring 
KEYWORD_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in KEYWORDS) + r")\b",
    re.IGNORECASE,
)


def load_and_filter_articles(path):
    """Load the raw article CSV and filter down to NWSL-relevant rows."""
    media = pd.read_csv(path)
    print(f"Total articles before filtering: {len(media)}")

    media["publish_date"] = pd.to_datetime(media["publish_date"])

    is_relevant = media["title"].apply(lambda title: bool(KEYWORD_PATTERN.search(str(title))))
    media = media[is_relevant]
    media = media.drop_duplicates(subset=["title", "publish_date"])
    media = media.drop(columns=["description"], errors="ignore")

    print(f"Total articles after filtering: {len(media)}")
    return media

In [2]:
media = load_and_filter_articles(os.path.join(DATA_RAW_DIR, "nwsl_articles_raw.csv"))

out_path = os.path.join(DATA_PROCESSED_DIR, "nwsl_articles_filtered.csv")
media[["publish_date", "media_name", "title", "url"]].to_csv(out_path, index=False)
print(f"Saved {len(media)} rows to nwsl_articles_filtered.csv")


Total articles before filtering: 59909
Total articles after filtering: 8328
Saved 8328 rows to nwsl_articles_filtered.csv


## Sentiment: positive vs. negative coverage split

VADER (`vaderSentiment`), scored on each article's title. Standard VADER thresholds: compound >= 0.05 is positive, <= -0.05 is negative, everything in between is neutral. Titles are short, so this is a coarse signal (a headline doesn't carry much sentiment-bearing language on its own), but it's enough to see the overall split and how it moves year to year.

In [3]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

media["sentiment_compound"] = media["title"].apply(lambda t: analyzer.polarity_scores(str(t))["compound"])

# Standard VADER cutoffs for compound score -> pos/neg/neutral label
sentiment_conditions = [
    media["sentiment_compound"] >= 0.05,
    media["sentiment_compound"] <= -0.05,
]
media["sentiment_label"] = pd.Series(
    pd.NA, index=media.index, dtype="object"
).mask(sentiment_conditions[0], "positive").mask(sentiment_conditions[1], "negative").fillna("neutral")

print("Sentiment split across all filtered articles:")
print(media["sentiment_label"].value_counts())
print()
print((media["sentiment_label"].value_counts(normalize=True) * 100).round(1).astype(str) + "%")

sentiment_out_path = os.path.join(DATA_PROCESSED_DIR, "nwsl_articles_with_sentiment.csv")
media[["publish_date", "media_name", "title", "url", "sentiment_compound", "sentiment_label"]].to_csv(
    sentiment_out_path, index=False
)
print(f"\nSaved {len(media)} rows to nwsl_articles_with_sentiment.csv")

Sentiment split across all filtered articles:
sentiment_label
positive    3803
neutral     3286
negative    1239
Name: count, dtype: int64

sentiment_label
positive    45.7%
neutral     39.5%
negative    14.9%
Name: proportion, dtype: object

Saved 8328 rows to nwsl_articles_with_sentiment.csv


## YouTube channel & video data

In [4]:
# 1. Channels: already one tidy row per channel, just get channel age
youtube_channels = pd.read_csv(os.path.join(DATA_RAW_DIR, "youtube_channels_raw.csv"))
# format="ISO8601": the API returns timestamps
youtube_channels["channel_created"] = pd.to_datetime(youtube_channels["channel_created"], format="ISO8601")
youtube_channels["snapshot_date"] = pd.to_datetime(youtube_channels["snapshot_date"])
youtube_channels["channel_age_years"] = (
    (youtube_channels["snapshot_date"] - youtube_channels["channel_created"].dt.tz_localize(None)).dt.days / 365.25
)
youtube_channels["views_per_video"] = youtube_channels["view_count"] / youtube_channels["video_count"]
# The API's own channel titles have stray whitespace on at least one row ("No White Shorts ")
youtube_channels["channel_name"] = youtube_channels["channel_name"].str.strip()

youtube_channels.to_csv(os.path.join(DATA_PROCESSED_DIR, "youtube_channels.csv"), index=False)
print(f"Saved {len(youtube_channels)} rows to youtube_channels.csv")
youtube_channels[["channel_name", "subscriber_count", "video_count", "channel_age_years"]]

Saved 6 rows to youtube_channels.csv


,channel_name,subscriber_count,video_count,channel_age_years
0,The Women's Game,96000,1345,2.600958
1,National Women's Soccer League,250000,6223,13.371663
2,RE,56100,630,6.392882
3,CBS Sports W Golazo,80900,5229,5.147159
4,No White Shorts,81700,152,6.088980
5,Just Women's Sports,300000,2248,7.288159


In [5]:
# 2. Videos: parse publish date, derive year/month, drop exact duplicate video_ids if they exist
youtube_videos = pd.read_csv(os.path.join(DATA_RAW_DIR, "youtube_videos_raw.csv"))
print(f"Total videos before cleaning: {len(youtube_videos)}")

youtube_videos["published_at"] = pd.to_datetime(youtube_videos["published_at"], format="ISO8601")
youtube_videos = youtube_videos.drop_duplicates(subset=["video_id"])
youtube_videos["publish_year"] = youtube_videos["published_at"].dt.year
youtube_videos["publish_month"] = youtube_videos["published_at"].dt.to_period("M").astype(str)

# Standardize with channel id
youtube_videos = youtube_videos.drop(columns=["channel_name"]).merge(
    youtube_channels[["channel_id", "channel_name"]], on="channel_id", how="left"
)

print(f"Total videos after cleaning: {len(youtube_videos)}")

youtube_videos.to_csv(os.path.join(DATA_PROCESSED_DIR, "youtube_videos.csv"), index=False)
print(f"Saved {len(youtube_videos)} rows to youtube_videos.csv")

Total videos before cleaning: 15825
Total videos after cleaning: 15825
Saved 15825 rows to youtube_videos.csv


/var/folders/2y/rz_rt3v15kv2qz95_7stn2_r0000gn/T/ipykernel_91532/3891969753.py:8: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  youtube_videos["publish_month"] = youtube_videos["published_at"].dt.to_period("M").astype(str)
